<a href="https://colab.research.google.com/github/Leashaniya/Research-Project/blob/leasha/1_5_clean_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
!pip -q install --upgrade openai


In [ ]:
# ===========================
# NOTEBOOK 1.5 — LLM CLEAN + REBUILD BLUEPRINTS (LLM-ALL-PAGES, RATE-LIMIT SAFE)
#
# Cleans:  cleaned_document.txt  -> cleaned_document_llm.txt
# Builds:  blueprint_llm.json + blueprint_with_subquestions_llm.json
# Keeps original files untouched.
#
# THIS VERSION:
# ✅ Sends ALL pages to LLM (after light rule-cleaning)
# ✅ Still has: throttling, retry with exponential backoff, caching by hash
# ✅ Skips papers already processed
# ✅ Caps input size per page to avoid huge token burns
# ===========================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install openai

import os, re, json, time, random, hashlib
from pathlib import Path
from openai import OpenAI

# -------------------------------
# CONFIG
# -------------------------------
BASE_DIR = Path("/content/drive/MyDrive/RP/text_extraction_hybrid")

MODEL = "gpt-4.1-mini"
MAX_PAPERS = None

# If you call LLM for ALL pages, set this safer (higher wait).
# If your account is slow/low quota, increase further (e.g., 25-35).
MIN_SECONDS_BETWEEN_CALLS = 22
MAX_RETRIES = 8

SKIP_IF_OUTPUT_EXISTS = True

# Cache file (so re-running doesn't call API again)
CACHE_PATH = BASE_DIR.parent / "llm_clean_cache.json"

# Safety cap for page size
MAX_CHARS_TO_SEND = 6500

# LLM mode: ALL pages (except tiny/huge)
CLEAN_MODE = "llm_all"  # "llm_all" only for your request

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = input("Paste OPENAI_API_KEY (will not be saved): ").strip()

client = OpenAI()

# -------------------------------
# REGEX: Page markers + Blueprint parsing
# -------------------------------
PAGE_MARK_RE = re.compile(r"---\s*PAGE\s*(\d+)\s*---", re.IGNORECASE)

Q_MAIN_RE = re.compile(r'.*\bQuestion\s*([0-9IVXLC]+)\b', re.IGNORECASE)
NUMERIC_MAIN_RE = re.compile(r'^\s*\(?\s*([0-9]+)\s*(?:[\.\)\-:])\s*', re.IGNORECASE)
ALT_Q_RE = re.compile(r'.*\bQ\s*[:\.]?\s*([0-9]+)\b', re.IGNORECASE)

MARKS_RE = re.compile(r'\(?\s*([0-9]{1,3})\s*marks?\s*\)?', re.IGNORECASE)
TOTAL_MARKS_PAREN_RE = re.compile(r"\(\s*(\d{1,3})\s*Marks?\s*\)", re.IGNORECASE)

FOOTER_RE = re.compile(r'Page\s+\d+\s*of\s*\d+', re.I)

def _looks_like_question_total(n: int) -> bool:
    return 10 <= n <= 100

def extract_total_marks_for_question(q_lines, prev_page_tail_lines):
    header_zone = "\n".join(q_lines[:12])
    m = TOTAL_MARKS_PAREN_RE.search(header_zone)
    if m:
        v = int(m.group(1))
        if _looks_like_question_total(v):
            return v

    prev_zone = "\n".join(prev_page_tail_lines[-10:]) if prev_page_tail_lines else ""
    m2 = TOTAL_MARKS_PAREN_RE.search(prev_zone)
    if m2:
        v = int(m2.group(1))
        if _looks_like_question_total(v):
            return v

    all_marks = [int(x) for x in MARKS_RE.findall("\n".join(q_lines))]
    s = sum(all_marks)
    if _looks_like_question_total(s):
        return s

    return None

def parse_blueprint_from_text(doc_text: str, pdf_stem: str, min_words=5):
    blueprint = []
    state = {"current_q": None, "q_buffer": [], "q_page": None}

    prev_page_tail = []
    current_page_lines = []
    page_no = None

    def finalize():
        if state["current_q"]:
            q_text = "\n".join(state["q_buffer"]).strip()
            if not q_text:
                return
            total_marks = extract_total_marks_for_question(state["q_buffer"], prev_page_tail)
            diag_refs = [d.strip() for d in re.findall(r'\[DIAGRAM:([^\]]+)\]', q_text)]
            blueprint.append({
                "question_id": str(state["current_q"]),
                "pdf_stem": pdf_stem,
                "page_no": state["q_page"],
                "marks": int(total_marks) if total_marks is not None else None,
                "text": q_text,
                "diagram_refs": diag_refs,
                "subquestions_full": [],
                "subquestions": []
            })
        state["current_q"] = None
        state["q_buffer"] = []

    for ln in doc_text.splitlines():
        ln_strip = ln.strip()

        mpage = PAGE_MARK_RE.search(ln_strip)
        if mpage:
            prev_page_tail = current_page_lines[-20:] if current_page_lines else prev_page_tail
            current_page_lines = []
            page_no = int(mpage.group(1))
            continue

        current_page_lines.append(ln_strip)

        if not ln_strip:
            if state["current_q"]:
                state["q_buffer"].append("")
            continue

        m_main = Q_MAIN_RE.match(ln_strip) or NUMERIC_MAIN_RE.match(ln_strip) or ALT_Q_RE.match(ln_strip)
        if m_main:
            finalize()
            qid = m_main.group(1)
            state["current_q"] = qid
            state["q_page"] = page_no
            state["q_buffer"] = [ln_strip]
            continue

        if state["current_q"]:
            state["q_buffer"].append(ln_strip)

    finalize()
    blueprint = [b for b in blueprint if len((b.get("text") or "").split()) >= min_words]
    return blueprint

# -------------------------------
# Page splitting / reassembly
# -------------------------------
def split_by_pages(doc_text: str):
    parts = []
    last_idx = 0
    matches = list(PAGE_MARK_RE.finditer(doc_text))
    if not matches:
        return [(None, doc_text, None)]

    for i, m in enumerate(matches):
        start = m.start()
        if i > 0:
            prev = matches[i-1]
            prev_page_no = int(prev.group(1))
            prev_marker = prev.group(0)
            body = doc_text[last_idx:start].strip("\n")
            parts.append((prev_page_no, body, prev_marker))
        last_idx = m.end()

    last = matches[-1]
    last_page_no = int(last.group(1))
    last_marker = last.group(0)
    body = doc_text[last_idx:].strip("\n")
    parts.append((last_page_no, body, last_marker))
    return parts

# -------------------------------
# Local (non-LLM) cleaning: cheap + safe
# -------------------------------
NOISE_LINE_RE = re.compile(r'^\s*([A-Za-z]{1,3}\d{0,3}|[\/\\\|\-_]{1,6}|[A-Za-z]{1,3})\s*$')
WEIRD_GARBAGE_RE = re.compile(r'[\uFFFD]+')

def rule_based_clean(text: str) -> str:
    lines = []
    for ln in text.splitlines():
        s = ln.strip()
        if not s:
            lines.append("")
            continue

        if "[DIAGRAM:" in s:
            lines.append(s)
            continue

        if FOOTER_RE.search(s):
            continue

        if NOISE_LINE_RE.match(s) and ("question" not in s.lower()) and ("marks" not in s.lower()):
            continue

        s = WEIRD_GARBAGE_RE.sub("", s)
        s = re.sub(r'\s+', ' ', s).strip()
        lines.append(s)

    out = "\n".join(lines)
    out = re.sub(r"\n{3,}", "\n\n", out).strip()
    return out

# -------------------------------
# Cache
# -------------------------------
def _hash_text(s: str) -> str:
    return hashlib.sha256((s or "").encode("utf-8", errors="ignore")).hexdigest()

if CACHE_PATH.exists():
    try:
        CACHE = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    except Exception:
        CACHE = {}
else:
    CACHE = {}

def cache_get(key: str):
    return CACHE.get(key)

def cache_set(key: str, value: str):
    CACHE[key] = value

def cache_flush():
    CACHE_PATH.write_text(json.dumps(CACHE, ensure_ascii=False, indent=2), encoding="utf-8")

# -------------------------------
# LLM call wrapper with throttle + retry
# -------------------------------
_last_call_ts = 0.0
llm_calls = 0  # accurate count: only increments when not cached

def _throttle():
    global _last_call_ts
    now = time.time()
    wait = MIN_SECONDS_BETWEEN_CALLS - (now - _last_call_ts)
    if wait > 0:
        time.sleep(wait)
    _last_call_ts = time.time()

def _get_output_text(resp) -> str:
    out = ""
    try:
        for item in getattr(resp, "output", []) or []:
            if getattr(item, "type", None) == "message":
                for c in getattr(item, "content", []) or []:
                    if getattr(c, "type", None) == "output_text":
                        out += getattr(c, "text", "") or ""
    except Exception:
        pass
    return (out or "").strip()

LLM_SYSTEM_RULES = (
    "Clean OCR exam text. Preserve meaning exactly. "
    "Keep numbering/marks unchanged. Keep [DIAGRAM: ...] exactly. "
    "Remove obvious OCR garbage. Do not invent content. Output only cleaned text."
)

def llm_clean_page(page_text: str) -> str:
    global llm_calls

    h = _hash_text(page_text)
    cached = cache_get(h)
    if cached is not None:
        return cached

    llm_calls += 1

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            _throttle()

            resp = client.responses.create(
                model=MODEL,
                instructions=LLM_SYSTEM_RULES,
                input=page_text,
                temperature=0.0,
                max_output_tokens=1200,
            )

            cleaned = _get_output_text(resp).strip()
            if not cleaned:
                cleaned = page_text.strip()

            cache_set(h, cleaned)
            if random.random() < 0.08:
                cache_flush()

            return cleaned

        except Exception as e:
            msg = str(e).lower()
            if ("rate limit" in msg) or ("429" in msg) or ("timeout" in msg) or ("server" in msg) or ("temporarily" in msg):
                backoff = min(180, (2 ** attempt) * 2) + random.uniform(0.0, 3.0)
                print(f"⚠️ LLM failed (attempt {attempt}/{MAX_RETRIES}). Backing off {backoff:.1f}s. Error: {e}")
                time.sleep(backoff)
                continue
            raise

    print("❌ All retries exhausted for a page; returning rule-based cleaned text.")
    cleaned = rule_based_clean(page_text)
    cache_set(h, cleaned)
    return cleaned

# -------------------------------
# RUN
# -------------------------------
paper_dirs = sorted([p for p in BASE_DIR.iterdir() if p.is_dir()])
if MAX_PAPERS is not None:
    paper_dirs = paper_dirs[:MAX_PAPERS]

print("Found paper folders:", len(paper_dirs))

processed = 0
skipped = 0
rule_only = 0

for pd in paper_dirs:
    in_path = pd / "cleaned_document.txt"
    if not in_path.exists():
        continue

    out_clean = pd / "cleaned_document_llm.txt"
    out_bp_main = pd / "blueprint_llm.json"
    out_bp_full = pd / "blueprint_with_subquestions_llm.json"

    if SKIP_IF_OUTPUT_EXISTS and out_clean.exists() and out_bp_main.exists() and out_bp_full.exists():
        skipped += 1
        continue

    pdf_stem = pd.name
    raw = in_path.read_text(encoding="utf-8", errors="ignore")
    pages = split_by_pages(raw)

    cleaned_pages = []
    for page_no, body, marker in pages:
        body = (body or "").strip()

        rb = rule_based_clean(body)

        # LLM ALL pages (except tiny/huge)
        use_llm = False
        if CLEAN_MODE == "llm_all":
            use_llm = (len(rb) >= 60) and (len(body) <= MAX_CHARS_TO_SEND)

        if use_llm:
            cleaned_body = llm_clean_page(rb)
        else:
            cleaned_body = rb
            rule_only += 1

        cleaned_pages.append((page_no, cleaned_body, marker))

    rebuilt = []
    for page_no, cleaned_body, marker in cleaned_pages:
        if marker:
            rebuilt.append(f"\n\n--- PAGE {page_no} ---\n")
        rebuilt.append(cleaned_body.strip())

    cleaned_doc = "\n".join([x for x in rebuilt if str(x).strip()]).strip()
    out_clean.write_text(cleaned_doc, encoding="utf-8")

    bp = parse_blueprint_from_text(cleaned_doc, pdf_stem)

    bp_main = []
    for q in bp:
        q2 = dict(q)
        q2.pop("subquestions_full", None)
        q2.pop("subquestions", None)
        bp_main.append(q2)

    out_bp_main.write_text(json.dumps(bp_main, indent=2, ensure_ascii=False), encoding="utf-8")
    out_bp_full.write_text(json.dumps(bp, indent=2, ensure_ascii=False), encoding="utf-8")

    marks = [q.get("marks") for q in bp_main]
    total = sum([m for m in marks if isinstance(m, int)])
    missing = sum([1 for m in marks if m is None])
    print(f"✅ {pdf_stem}: questions={len(bp_main)} total_known={total} missing_marks={missing}")

    processed += 1

cache_flush()

print("\nDONE.")
print("Processed papers :", processed)
print("Skipped papers   :", skipped)
print("LLM API calls    :", llm_calls, "(cached pages not counted)")
print("Rule-only pages  :", rule_only)
print("\nNext: point Notebook 2 to read blueprint_with_subquestions_llm.json (or blueprint_llm.json).")
print(f"Cache saved at: {CACHE_PATH}")
